   ... expected to know how to use CBuild/`cb`, plus normal C tools like `gcc`,
   `gdb` and `valgrind`. The PDF also says compilation/crashing penalties can
   be harsh, so clean compiling code matters a lot.

   Source context...

---
KEY CONCEPT
   Raw compiler command:
   
```c
gcc -std=c17 -Wall -Wpedantic -o prog prog.c
```
   means: manually tell GCC flags, output names, and source files.

   CBuild/`cb` is closer to:

```c
cb
```
   meaning: read the provided build config, work out what to compile/link, then
   call the compiler for you.

   So mentally:
```
gcc/clang = direct compiler command
make      = build automation using Makefile rules
cb        = course build automation using provided .cbuild/.build config
```
   You do not need to write pattern rules. You mainly need to know:
```bash
cb
cb clean
```
   and understand that GCC warnings/errors still matter because `cb` eventually
   invokes GCC/Clang underneath. 


---

   Use the `wc` command to count the number of lines, words, and bytes in the 
   files specified by the File parameter. If a file is not specified for the
   File parameter, standard input is used.

```sh
cb
cb --clean
cb --allclean
cb --test
cb --install
```

   Locally `cb` is not on this machine's PATH... 

---
CONCEPTS
   CBuild/`cb` is basically "Course Make without writing Makefiles."

   In `08.cbuild/test1/.cbuild`, the whole config is:
```make
BUILD = avgwordlen testlist
```
   That means: by default, build two executable programs, `avgwordlen` and
   `testlist`.

   `cb` scans the ``.c` and `.h` files, especially local includes like:

```c
#include "intlist.h"
```
   Then it infers dependencies:

```
testlist.c includes intlist.h
intlist.h has matching intlist.c
therefore testlist needs testlist.o + intlist.o
```
   The raw GCC equivalent would be:
```sh
gcc -Wall -c intlist.c
gcc -Wall -c testlist.c
gcc testlist.o intlist.o -o testlist
```

   ... But wth `cb`, you just do:
```sh
cb
```

   Important distinction:
```
compiler error = bad C syntax/types in one source file
linker error   = object files compiled, but final executable cannot be connected
runtime crash  = program built, then segfaulted/asserted/etc
```

...
```c
intlist_append(xs, 3);
```
   Linker error if `intlist_append` is declared in `.h` but not implemented in 
   `.c`.
```
int *p = NULL;
*p = 3;
```
   Runtime crash..

   For clean builds:
```sh
cb --clean
```
   removes generated objects/executables.

```sh
cb --allclean
```
   cleans and rebuilds.

   This matters in exams because stale object files can hide problems, and the
   hints PDF says non-compiling/crashing tasks get heavy penalties.

---

Q1

```sh
cb
```

---
Q2

   Conceptual hint: `avgwordlen.c` and `testlist.c` are both main programs.

   Syntax hints: multiple `main` functions cannot be linked into one executable.

   ANSWER: It tries to compile/link every `.c` file into one program. If more
   than one file has `main`, linking fails with a multiple-definition error.


---
Q3

   Manually, what two-stage GCC process builds `testlist` from `testlist.c`
   and `intlist.c`?

   Conceptual hint: first compile `.c -> .o`, then link `.o -> executable`.

   Syntax hint: use `-c` for compile-only.

```sh
gcc -Wall -c testlist.c
gcc -Wall -Wextra -c intlist.c
gcc testlist.o intlist.o -o testlist
```

---
Q4
   `testlist.c` includes `intlist.h`. There is also an `intlist.c`. What does
   `cb` infer?

   CONCEPTUAL HINT: matching `.h` and `.c` form a module.

   Answer: `testlist` depends on the `intlist` module, so `cb` should compile 
   and link `intlist.o` into `testlist`.

---
Q5
   ... implementation changed, not interface.

   ANSWER: `intlist.o` should recompile, and programs using it, such as 
   `testlist` and `avgwordlen`, should relink.

---
Q6
   ... interface changed. ... Anything including `intlist.h` should recompile,
   so likely `intlist.o`, `testlist.o`, `avgwordlen.o`, then the executables
   relink.

---
Q7
```sh
undefined reference to `intlist_length`
```
   ... This is link error...

   ANSWER: Linker error. The function was declared or called, but no matching
   implementation was found during linking. 

---
Q8
```
warning: implicit declaration of function `foo`
```
   CONCEPTUAL HINT: Compiler has not seen the function prototype.

   ANSWER: Missing header include, misspelled function name, or no declaration
   before use. In C17 with strict flags, treat this as serious.


